# T-Rex backbone/checkpoint downloader (for a server with no HF hub access)

Run this on Colab (no GPU needed -- CPU runtime is enough, this is a
download-and-zip job). It fetches the same two assets
`scripts/colab_setup.sh` would download directly on a Colab session with
internet access, bundles them into one zip, and saves that zip to Drive.

Copy the zip from Drive to your server, then run (on the server):

```bash
BACKBONE_ZIP=/path/to/trex_backbones_ckpts.zip bash scripts/setup_server_env.sh
```

Add more repos to `BACKBONES` below if a future run needs another checkpoint
(e.g. a different midtrain epoch, or a separately-hosted DeformEncoder/VQ-VAE
checkpoint) -- everything downstream (the zip step, the server unpack step)
already generalizes to N entries, not just these two.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install -q -U huggingface_hub


In [ ]:
import os

ASSET_ROOT = "/content/backbones"
DRIVE_OUT_DIR = "/content/drive/MyDrive/iros2026"   # matches trex_origami/README.md's Colab layout
ZIP_NAME = "trex_backbones_ckpts.zip"

os.makedirs(ASSET_ROOT, exist_ok=True)
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

# (repo_id, target subdir under ASSET_ROOT, repo_type, "existence marker" file
#  used to skip a re-download if this cell is re-run).
BACKBONES = [
    ("Qwen/Qwen3-VL-2B-Instruct", "Qwen3-VL-2B-Instruct", "model", "config.json"),
    ("miniFranka/T-Rex_midtrain_mecka23k_ucb100_vqvae_epoch6", "trex_midtrain", "model", None),
]


In [ ]:
import glob
from huggingface_hub import snapshot_download

for repo_id, subdir, repo_type, marker in BACKBONES:
    target = os.path.join(ASSET_ROOT, subdir)
    if marker and os.path.exists(os.path.join(target, marker)):
        print(f"[skip] {repo_id} already at {target}")
        continue
    if marker is None and glob.glob(os.path.join(target, "**", "model.pt"), recursive=True):
        print(f"[skip] {repo_id} already at {target} (found model.pt)")
        continue
    print(f"[download] {repo_id} -> {target}")
    snapshot_download(repo_id, repo_type=repo_type, local_dir=target)

print("done.")


In [ ]:
# Same checks scripts/colab_setup.sh prints -- read these before zipping.
# If vqvae_config is MISSING, STOP: that's the ablation the T-Rex paper shows
# costs 65% -> 45% success (untrained tactile expert). Re-download instead of
# shipping a bundle with this problem to the server.
import json

ckpt_dirs = glob.glob(os.path.join(ASSET_ROOT, "trex_midtrain", "**", "model.pt"), recursive=True)
assert ckpt_dirs, "no model.pt found under trex_midtrain -- download failed"
ckpt = os.path.dirname(ckpt_dirs[0])
print("midtrain checkpoint dir:", ckpt)

args_path = os.path.join(ckpt, "training_args.json")
if os.path.exists(args_path):
    ta = json.load(open(args_path))
    print("  use_tactile_vqvae:", ta.get("use_tactile_vqvae"),
          "| vqvae_config:", "present" if ta.get("vqvae_config") else "MISSING <-- STOP if this says MISSING",
          "| action_dim:", ta.get("action_dim"), "| action_chunk:", ta.get("action_chunk"))
else:
    print("  WARNING: no training_args.json next to model.pt")

assert os.path.exists(os.path.join(ASSET_ROOT, "Qwen3-VL-2B-Instruct", "config.json")), \
    "base model config.json missing"
print("base model OK:", os.path.join(ASSET_ROOT, "Qwen3-VL-2B-Instruct"))


In [ ]:
# Zip ASSET_ROOT's contents at its root (Qwen3-VL-2B-Instruct/ and
# trex_midtrain/ as top-level entries) -- setup_server_env.sh on the server
# expects that layout, matching what colab_setup.sh would have produced
# directly under ASSET_ROOT.
zip_path = os.path.join("/content", ZIP_NAME)
!cd {ASSET_ROOT} && zip -qr {zip_path} .
print("zipped:", zip_path)
!ls -la {zip_path}


In [ ]:
import shutil
dest = os.path.join(DRIVE_OUT_DIR, ZIP_NAME)
shutil.copy(zip_path, dest)
print("saved to Drive:", dest)
print()
print("On the server:")
print(f"  BACKBONE_ZIP=/path/to/{ZIP_NAME} bash scripts/setup_server_env.sh")
